In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import re
import nltk
from nltk.corpus import stopwords

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GRU, Dense, Dropout
from sklearn.model_selection import train_test_split

nltk.download("stopwords")

df = pd.read_csv("IMDB Dataset.csv")

df.head()


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [2]:
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)         # remove HTML
    text = re.sub(r"[^a-zA-Z\s]", "", text)   # remove punctuation/digits
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df["clean_review"] = df["review"].apply(clean_text)

MAX_WORDS = 10000     # số từ lấy trong từ điển
MAX_LEN = 200         # padding mỗi câu về 200 từ

tokenizer = Tokenizer(num_words=MAX_WORDS)
tokenizer.fit_on_texts(df["clean_review"])

sequences = tokenizer.texts_to_sequences(df["clean_review"])
X = pad_sequences(sequences, maxlen=MAX_LEN)

# label: positive -> 1, negative -> 0
y = df["sentiment"].map({"positive":1, "negative":0}).values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [3]:
model_lstm = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=128, input_length=MAX_LEN),
    LSTM(128, return_sequences=False),
    Dropout(0.4),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model_lstm.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
model_lstm.summary()

model_gru = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=128, input_length=MAX_LEN),
    GRU(128),
    Dropout(0.4),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

model_gru.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
model_gru.summary()

history = model_lstm.fit(
    X_train, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2
)

history = model_gru.fit(
    X_train, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2
)

loss_lstm, acc_lstm = model_lstm.evaluate(X_test, y_test)
print("Độ chính xác của mô hình LSTM trên tập test:", acc_lstm)

loss_gru, acc_gru = model_gru.evaluate(X_test, y_test)
print("Độ chính xác của mô hình GRU trên tập test:", acc_gru)

C:\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ gru (GRU)                            │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 100s 393ms/step - accuracy: 0.8239 - loss: 0.3836 - val_accuracy: 0.8764 - val_loss: 0.3021
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 143s 398ms/step - accuracy: 0.9126 - loss: 0.2259 - val_accuracy: 0.8702 - val_loss: 0.3072
Epoch 3/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 138s 383ms/step - accuracy: 0.9386 - loss: 0.1672 - val_accuracy: 0.8735 - val_loss: 0.3495
Epoch 4/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 142s 382ms/step - accuracy: 0.9538 - loss: 0.1268 - val_accuracy: 0.8606 - val_loss: 0.4111
Epoch 5/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 93s 372ms/step - accuracy: 0.9655 - loss: 0.0992 - val_accuracy: 0.8576 - val_loss: 0.4488
Epoch 1/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 90s 352ms/step - accuracy: 0.7772 - loss: 0.4486 - val_accuracy: 0.8700 - val_loss: 0.3145
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 142s 351ms/step - accuracy: 0.9068 - loss: 0.2430 - val_accuracy: 0.8810 - val_loss: 0.2924
Epoch 3/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 144s 357ms/step - accuracy: 0.9338 - loss: 0.1

In [4]:
def predict_sentiment_lstm(sentence):
    text = clean_text(sentence)
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=MAX_LEN)
    pred = model_lstm.predict(pad)[0][0]
    return "Positive" if pred >= 0.5 else "Negative"

def predict_sentiment_gru(sentence):
    text = clean_text(sentence)
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=MAX_LEN)
    pred = model_gru.predict(pad)[0][0]
    return "Positive" if pred >= 0.5 else "Negative"

review1 = "The movie was incredibly good!"
review2 = "Worst movie I have ever seen"

print(predict_sentiment_lstm(review1))
print(predict_sentiment_lstm(review2))
print(predict_sentiment_gru(review1))
print(predict_sentiment_gru(review2))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 188ms/step
Positive
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
Negative
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 215ms/step
Positive
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
Negative
